# Random Forest — Optuna hyperparameter study on `fe_v4_native` (CPU, 12 h)

Runs an 80-trial Optuna study for `RandomForestClassifier` on the **`fe_v4_native`**
feature set — the same minimal engineered features the top GBDT runs use:
`AverageMonthly`, `contract_x_payment`, `contract_x_internet` on top of the 19 base
native-categorical features (22 total).

**Why ordinal encoding?** `sklearn`'s `RandomForestClassifier` has no native
categorical path: it rejects `category`-dtype columns. Each categorical column is
converted to its `cat.codes` integer (0, 1, 2, …) before training. This is leakage-free
because `cat.codes` depends only on the categories defined in the dtype — which come from
`prepare_data` on the full dataset — not on any per-fold statistic.

**Why CPU?** RF is embarrassingly parallel across trees (`n_jobs=-1`), already saturates
the instance's vCPUs, and gets no benefit from a GPU accelerator. CPU quota is effectively
unlimited compared to GPU hours.

**Settings (right sidebar):** Accelerator → **None / CPU**; Internet → **On**; Add
Input → **playground-series-s6e3**.

> ⏱️ Each trial runs 3-fold inner CV with `n_jobs=-1` inside the RF. Typical trial time
> ~3–15 min depending on `n_estimators` and `max_depth`; the TPE sampler tends to explore
> smaller trees first so early trials are faster. **SMOKE-TEST FIRST:** set `n_trials=4`
> to measure per-trial time before committing to the full 80-trial Save & Run All.
> Expect the full study to take 8–12 h.

In [1]:
# List attached inputs (confirm the competition data is mounted).
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e3/train.csv
/kaggle/input/competitions/playground-series-s6e3/test.csv


In [2]:
import os, sys, subprocess

REPO_URL  = "https://github.com/biswajit-nag/Predict-Customer-Churn.git"
REPO_ROOT = "/kaggle/working/Predict-Customer-Churn"

if not os.path.exists(REPO_ROOT):
    subprocess.run(["git", "clone", REPO_URL, REPO_ROOT], check=True)

os.chdir(REPO_ROOT)                       # CWD = repo root (fixes data paths + git_info)
# Put the clone FIRST on sys.path so its `src` wins over any other module named `src`,
# and drop a possibly-stale `src` cached by an earlier cell.
sys.path.insert(0, REPO_ROOT)
for _m in [k for k in list(sys.modules) if k == "src" or k.startswith("src.")]:
    del sys.modules[_m]
print("CWD:", os.getcwd())

Cloning into '/kaggle/working/Predict-Customer-Churn'...


CWD: /kaggle/working/Predict-Customer-Churn


Updating files: 100% (319/319), done.


In [3]:
# Kaggle's image ships sklearn; only optuna may be missing.
!pip install -q optuna

import sklearn, optuna
print("sklearn:", sklearn.__version__)
print("optuna: ", optuna.__version__)
print("CPU cores:", os.cpu_count())

# Quick sanity fit to confirm RandomForestClassifier works before the full study.
import numpy as np
from sklearn.ensemble import RandomForestClassifier

_Xs = np.random.rand(2000, 6)
_ys = (np.random.rand(2000) > 0.5).astype(int)
RandomForestClassifier(n_estimators=5, max_depth=5, random_state=42).fit(_Xs, _ys)
print("RandomForestClassifier CPU OK")

sklearn: 1.6.1
optuna:  4.8.0
CPU cores: 4
RandomForestClassifier CPU OK


In [4]:
# data/processed/*.parquet are git-ignored, so absent from the clone. Rebuild the
# NATIVE (category-dtype) frames from the attached competition CSVs — same data path
# as all the GBDT fe_v4 runs.
import shutil
from pathlib import Path

raw_dir = Path(REPO_ROOT) / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)
for f in ("train.csv", "test.csv"):
    shutil.copy(f"/kaggle/input/competitions/playground-series-s6e3/{f}", raw_dir / f)

from src.data import prepare_data
train_df, test_df = prepare_data(encoding='native', force=True)
print(f'Loaded native: train_df {train_df.shape}, test_df {test_df.shape}')

Preprocessed and saved (native): train_df (594194, 21), test_df (254655, 20)
Loaded native: train_df (594194, 21), test_df (254655, 20)


In [5]:
import json
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

from src.tracking import DATA_DIR, RUNS_DIR, RUNS_CSV
from src.cv import run_cv_experiment, save_experiment

### Feature engineering — `fe_v4_native` (minimal set)

Identical to the GBDT min3 / TabICL runs: `AverageMonthly` plus the two
low-cardinality crosses, all row-wise (stateless) so there is no leakage when
applied before the CV split. No `tenure == 0` rows exist in the data, so
`AverageMonthly` is always finite.

In [6]:
DATA_VERSION = 'fe_v4_native'   # minimal engineered set on the native categorical base


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Row-wise (stateless) FE — identical to the GBDT min3 / TabICL runs."""
    df = df.copy()
    df['AverageMonthly'] = df['TotalCharges'] / df['tenure']
    df['contract_x_payment'] = (df['Contract'].astype(str) + ' | '
                                + df['PaymentMethod'].astype(str)).astype('category')
    df['contract_x_internet'] = (df['Contract'].astype(str) + ' | '
                                 + df['InternetService'].astype(str)).astype('category')
    return df


# Cache engineered parquets per DATA_VERSION (same convention as Experiments.ipynb).
fe_train_path = DATA_DIR / f'train_df_{DATA_VERSION}.parquet'
fe_test_path  = DATA_DIR / f'test_df_{DATA_VERSION}.parquet'

if fe_train_path.exists() and fe_test_path.exists():
    train_df = pd.read_parquet(fe_train_path)
    test_df  = pd.read_parquet(fe_test_path)
    print(f'Loaded cached FE: {DATA_VERSION}')
else:
    train_df = engineer_features(train_df)
    test_df  = engineer_features(test_df)
    pq.write_table(pa.Table.from_pandas(train_df, preserve_index=False), fe_train_path)
    pq.write_table(pa.Table.from_pandas(test_df,  preserve_index=False), fe_test_path)
    print(f'Computed and cached FE: {DATA_VERSION}')

Loaded cached FE: fe_v4_native


In [7]:
encoded_features = [c for c in train_df.columns if c not in ('id', 'Churn')]
X_train_base = train_df[encoded_features]
y_train      = train_df['Churn']
X_test_base  = test_df[encoded_features]
print(f'Base: X_train {X_train_base.shape}  X_test {X_test_base.shape}  features: {len(encoded_features)}')

# --- Ordinal-encode categorical columns for RandomForestClassifier ---
# sklearn's RF rejects category-dtype columns. cat.codes converts each level to its
# integer position (0, 1, 2, …); unknown test levels become -1 (treated by RF as a
# split boundary, not as missing in a harmful sense). Alignment via set_categories
# guarantees train and test use identical codes for shared levels.
cat_cols = [c for c in encoded_features
            if isinstance(X_train_base[c].dtype, pd.CategoricalDtype)]
print(f'Ordinal-encoding {len(cat_cols)} categorical columns: {cat_cols}')

X_train = X_train_base.copy()
X_test  = X_test_base.copy()
for col in cat_cols:
    X_test[col]  = X_test[col].cat.set_categories(X_train[col].cat.categories)
    X_train[col] = X_train[col].cat.codes
    X_test[col]  = X_test[col].cat.codes

print(f'After encoding: dtypes = {dict(X_train.dtypes.value_counts())}')

Base: X_train (594194, 22)  X_test (254655, 22)  features: 22
Ordinal-encoding 17 categorical columns: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'contract_x_payment', 'contract_x_internet']
After encoding: dtypes = {dtype('int8'): np.int64(17), dtype('float64'): np.int64(3), dtype('int64'): np.int64(2)}


### Optuna study — Random Forest on `fe_v4_native`

Searches the main RF knobs with TPE (3-fold inner CV, ROC-AUC). Each trial runs
`RandomForestClassifier` with `n_jobs=-1` (all vCPUs for tree building) sequentially
— one trial at a time — which avoids nested thread-pool contention that would arise
from running `N_JOBS` parallel trials each with their own multi-threaded RF.

| Hyperparameter | Range | Notes |
|---|---|---|
| `n_estimators` | 100–600 | More trees = better ensembling; 600 caps wall time |
| `max_depth` | 5–30 | Uncapped trees on 594 k rows are prohibitively slow |
| `min_samples_split` | 2–50 | Minimum node size to attempt a split |
| `min_samples_leaf` | 1–25 | Minimum samples per leaf |
| `max_features` | `sqrt`, `log2`, float [0.05, 0.7] | Float branch lets TPE explore denser fractions |
| `max_samples` | 0.4–1.0 | Bootstrap fraction — controls variance/bias trade-off |

Trial 0 is warm-started from sklearn's canonical RF defaults (`sqrt` features, no depth
limit → capped to `max_depth=15` here to stay within the search space).

> **SMOKE-TEST:** Change `n_trials=4` first to measure per-trial time, then restore
> `n_trials=80` and Save & Run All. A full 80-trial run should complete within 12 h.

In [8]:
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

rf_inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)


def rf_objective(trial):
    # max_features: two named shortcuts plus a searched float fraction.
    # Storing type and value separately lets TPE model each branch independently.
    mf_choice = trial.suggest_categorical('max_features_type', ['sqrt', 'log2', 'float'])
    if mf_choice == 'float':
        max_features = trial.suggest_float('max_features_float', 0.05, 0.7)
    else:
        max_features = mf_choice

    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 600),
        'max_depth':         trial.suggest_int('max_depth', 5, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 50),
        'min_samples_leaf':  trial.suggest_int('min_samples_leaf', 1, 25),
        'max_features':      max_features,
        'max_samples':       trial.suggest_float('max_samples', 0.4, 1.0),
        'n_jobs':            -1,      # parallelize tree building across all vCPUs
        'random_state':      42,
    }
    scores = cross_val_score(
        RandomForestClassifier(**params), X_train, y_train,
        cv=rf_inner_cv, scoring='roc_auc',
        n_jobs=1,  # one trial at a time; RF already uses all cores via n_jobs=-1
    )
    return scores.mean()


rf_study = optuna.create_study(
    study_name='rf-fe_v4_native',
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
)

# Warm-start from sklearn's canonical RF defaults: sqrt features, moderate depth.
rf_study.enqueue_trial({
    'n_estimators':      200,
    'max_depth':         15,
    'min_samples_split': 2,
    'min_samples_leaf':  1,
    'max_features_type': 'sqrt',
    'max_samples':       1.0,
})

# SMOKE-TEST: set n_trials=4 first to confirm per-trial timing, then restore 80.
rf_study.optimize(rf_objective, n_trials=80, show_progress_bar=True)

print(f'Best inner-CV ROC AUC: {rf_study.best_value:.6f}  (trial {rf_study.best_trial.number})')
for k, v in rf_study.best_params.items():
    print(f'  {k:25s} {v}')

  0%|          | 0/80 [00:00<?, ?it/s]

Best inner-CV ROC AUC: 0.914483  (trial 78)
  max_features_type         float
  max_features_float        0.3842671995295461
  n_estimators              572
  max_depth                 21
  min_samples_split         4
  min_samples_leaf          19
  max_samples               0.41582071429201733


### Run configuration

Reconstructs `max_features` from the split Optuna params before passing to
`RandomForestClassifier`. The ordinal-encoded `X_train` / `X_test` are passed to
`run_cv_experiment`; `data_version` still points to the underlying `fe_v4_native`
parquet (the ordinal encoding is an in-memory transformation, not a new parquet).

In [9]:
# Reconstruct max_features from the split Optuna params.
_mf_type  = rf_study.best_params.get('max_features_type')
_mf_float = rf_study.best_params.get('max_features_float')
_best_max_features = _mf_float if _mf_type == 'float' else _mf_type

_rf_params = {k: v for k, v in rf_study.best_params.items()
              if k not in ('max_features_type', 'max_features_float')}
_rf_params['max_features'] = _best_max_features
_rf_params['n_jobs']       = -1
_rf_params['random_state'] = 42

print('Best RF params:')
for k, v in _rf_params.items():
    print(f'  {k:22s} {v}')

run_config = {
    'model_factory': lambda params: RandomForestClassifier(**params),
    'params':        _rf_params,
    'metric':        accuracy_score,
    'metric_name':   'accuracy',
    'cv':            StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    'tag':           'rf-optuna-fe_v4_native',
    'notes': (
        'RandomForestClassifier on fe_v4_native (AverageMonthly + contract_x_payment + '
        'contract_x_internet + 19 base native categoricals). Category-dtype columns '
        'ordinal-encoded via cat.codes before training (train/test aligned with '
        'set_categories). Best params from an 80-trial Optuna study (3-fold inner CV, '
        'ROC-AUC, TPE); trial 0 seeded from sklearn canonical RF defaults. '
        'Data regenerated on-platform; data_hash differs from local runs. '
        'Notebook: kaggle/predict-customer-churn-rf-cpu-fe_v4.ipynb.'
    ),
    'parent_run_id': '',
    'save_models':   False,
    'data_version':  DATA_VERSION,
}

Best RF params:
  n_estimators           572
  max_depth              21
  min_samples_split      4
  min_samples_leaf       19
  max_samples            0.41582071429201733
  max_features           0.3842671995295461
  n_jobs                 -1
  random_state           42


In [10]:
# Step 1 — Run the experiment (fits 5 folds, prints OOF accuracy + ROC-AUC).
# X_train / X_test here are the ordinal-encoded versions built in the feature-setup cell.
result = run_cv_experiment(run_config, X_train, y_train, X_test, encoded_features)

Run ID: 20260612-133154-c3656f
Tag:    rf-optuna-fe_v4_native

Fold 0: accuracy=0.8598  roc_auc=0.9142  (fit 134.7s)
Fold 1: accuracy=0.8602  roc_auc=0.9153  (fit 138.9s)
Fold 2: accuracy=0.8595  roc_auc=0.9147  (fit 137.6s)
Fold 3: accuracy=0.8610  roc_auc=0.9159  (fit 135.2s)
Fold 4: accuracy=0.8595  roc_auc=0.9133  (fit 134.8s)

OOF accuracy: 0.8600
OOF ROC-AUC:  0.9147
Folds:        0.8600 ± 0.0006

Run complete. Call save_experiment(result) to log this run permanently.


In [11]:
# Step 2 — Save the run (review the OOF ROC-AUC above first).
run_id = save_experiment(result)

Saved to: /kaggle/working/Predict-Customer-Churn/experiments/runs/20260612-133154-c3656f


### Build a submission (optional)

`test_proba_mean` is the fold-bagged (5 folds) churn probability for the full test
set. The competition metric is ROC-AUC, so submit the probability directly.

In [12]:
submission = pd.DataFrame({
    'id':    test_df['id'],
    'Churn': result['artifacts']['test_proba_mean'],
})
submission.to_csv('/kaggle/working/submission.csv', index=False)
print(submission.head())
print('wrote /kaggle/working/submission.csv', submission.shape)

       id     Churn
0  594194  0.048532
1  594195  0.000056
2  594196  0.092842
3  594197  0.002810
4  594198  0.541599
wrote /kaggle/working/submission.csv (254655, 2)


### Bundle run artifacts + source notebook into one zip

Zips the run directory, `runs.csv`, and the source `.ipynb` into a single archive
on the Output tab. To fold the run back into the local repo, follow
**§7–8 of `docs/kaggle_gpu_workflow.md`**.

In [13]:
import shutil
from pathlib import Path
from src.tracking import RUNS_DIR, RUNS_CSV

BUNDLE = Path('/kaggle/working/bundle')
if BUNDLE.exists():
    shutil.rmtree(BUNDLE)

# 1) heavy run artifacts (params, oof_proba, test_proba_*, metrics, env, git diff)
shutil.copytree(RUNS_DIR / run_id, BUNDLE / 'runs' / run_id)
# 2) the master index row
shutil.copy(RUNS_CSV, BUNDLE / 'runs.csv')
# 3) source notebook committed in the cloned repo
src_nb = Path(REPO_ROOT) / 'kaggle' / 'predict-customer-churn-rf-cpu-fe_v4.ipynb'
if src_nb.exists():
    shutil.copy(src_nb, BUNDLE / src_nb.name)
    print('bundled notebook:', src_nb.name)
else:
    print('source notebook not found in clone (push it to master first for future runs)')

archive = shutil.make_archive(f'/kaggle/working/{run_id}_bundle', 'zip', BUNDLE)
print('wrote', archive)

source notebook not found in clone (push it to master first for future runs)
wrote /kaggle/working/20260612-133154-c3656f_bundle.zip
